In [1]:
# Sensitivity analysis: outcome threshold 130/80 instead of 140/90 (planned in D-003).
# Same population, predictors, models and rules as the main analysis; only the outcome changes.
import sys
from pathlib import Path
import numpy as np
import pandas as pd
pd.set_option("display.width", 200)

def find_project_root():
    path = Path.cwd().resolve()
    for candidate in [path, *path.parents]:
        if (candidate / "src" / "modeling.py").is_file():
            return candidate
    raise FileNotFoundError("Could not find src/modeling.py from " + str(path))

sys.path.insert(0, str(find_project_root() / "src"))
import importlib
import modeling as m
importlib.reload(m)

Y = "outcome_130_80"
dev = m.load("2017-2020")
val = m.load("2021-2023")
y_dev = dev[Y].to_numpy()
y_val = val[Y].to_numpy()
print(f"Positive at 130/80: development {y_dev.mean():.1%}, validation {y_val.mean():.1%}\n")

candidates = {"Age only": ["age"], "Model A": m.MODEL_A, "Model B": m.MODEL_B}
COLS = ["roc_auc", "cal_slope", "obs_exp", "sensitivity", "specificity",
        "referred_pct", "screen_per_case"]

# ---- 1. Internal validation (repeated CV on 2017-2020) ----
cv_rows, thresholds = {}, {}
for name, predictors in candidates.items():
    res = m.cross_validate(m.make_logistic, predictors, dev, outcome=Y)
    cv_rows[name] = m.summarise(res).loc[COLS, "mean"]
    thresholds[name] = res["thresholds"]["threshold"].mean()
print("=== 1. Cross-validation 2017-2020 (outcome 130/80) ===")
print(pd.DataFrame(cv_rows).T.round(3), "\n")

# ---- 2. Temporal validation on 2021-2023 (fit on all 2017-2020, fixed thresholds) ----
val_rows = {}
for name, predictors in candidates.items():
    model = m.make_logistic(predictors).fit(dev[predictors], y_dev)
    p = model.predict_proba(val[predictors])[:, 1]
    r = m.metrics(y_val, p)
    r.update(m.at_threshold(y_val, p, thresholds[name]))
    val_rows[name] = pd.Series(r)[COLS]
age_cut = m.threshold_for_sensitivity(y_dev, dev["age"].to_numpy())
rule = m.at_threshold(y_val, val["age"].to_numpy(), age_cut)
rule["roc_auc"] = m.roc_auc_score(y_val, val["age"])
val_rows[f"Age rule (>= {age_cut:.0f})"] = pd.Series(rule).reindex(COLS)
print("=== 2. Temporal validation 2021-2023 (outcome 130/80) ===")
print(pd.DataFrame(val_rows).T.round(3))

Positive at 130/80: development 33.4%, validation 31.7%

=== 1. Cross-validation 2017-2020 (outcome 130/80) ===
          roc_auc  cal_slope  obs_exp  sensitivity  specificity  referred_pct  screen_per_case
Age only    0.676      0.993      1.0        0.802        0.440        64.064            2.394
Model A     0.701      0.971      1.0        0.801        0.493        60.483            2.264
Model B     0.708      0.959      1.0        0.801        0.511        59.263            2.218 

=== 2. Temporal validation 2021-2023 (outcome 130/80) ===
                  roc_auc  cal_slope  obs_exp  sensitivity  specificity  referred_pct  screen_per_case
Age only            0.612      0.620    0.877        0.800        0.349        69.808            2.752
Model A             0.660      0.768    0.896        0.821        0.413        66.100            2.541
Model B             0.671      0.784    0.894        0.809        0.438        64.037            2.497
Age rule (>= 36)    0.612        NaN